In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [40]:
df=pd.read_csv("data - data.csv")

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [42]:
df

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,1.184,2.776,3.001,1.471,...,25.380,17.33,184.60,2019.0,1.622,6.656,7.119,2.654,4.601,1.189
1,842517,M,20.57,17.77,132.90,1326.0,8.474,7.864,869.000,7.017,...,24.990,23.41,158.80,1956.0,1.238,1.866,2.416,186.000,275.000,8.902
2,84300903,M,19.69,21.25,130.00,1203.0,1.096,1.599,1.974,1.279,...,23.570,25.53,152.50,1709.0,1.444,4.245,4.504,243.000,3.613,8.758
3,84348301,M,11.42,20.38,77.58,386.1,1.425,2.839,2.414,1.052,...,14.910,26.50,98.87,567.7,2.098,8.663,6.869,2.575,6.638,173.000
4,84358402,M,20.29,14.34,135.10,1297.0,1.003,1.328,198.000,1.043,...,22.540,16.67,152.20,1575.0,1.374,205.000,0.400,1.625,2.364,7.678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,111.000,1.159,2.439,1.389,...,25.450,26.40,166.10,2027.0,141.000,2.113,4.107,2.216,206.000,7.115
565,926682,M,20.13,28.25,131.20,1261.0,978.000,1.034,144.000,9.791,...,23.690,38.25,155.00,1731.0,1.166,1.922,3.215,1.628,2.572,6.637
566,926954,M,16.60,28.08,108.30,858.1,8.455,1.023,9.251,5.302,...,18.980,34.12,126.70,1124.0,1.139,3.094,3.403,1.418,2.218,782.000
567,927241,M,20.60,29.33,140.10,1265.0,1.178,277.000,3.514,152.000,...,25.740,39.42,184.60,1821.0,165.000,8.681,9.387,265.000,4.087,124.000


In [43]:
df["diagnosis"].value_counts() # B=%63,M=%37 hafif bir dengesizlik

diagnosis
B    357
M    212
Name: count, dtype: int64

In [44]:
X=df.drop(["diagnosis","id"],axis=1)
y = df["diagnosis"]

In [45]:
from sklearn.model_selection import train_test_split

In [46]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [47]:
#from sklearn.preprocessing import StandardScaler
#scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train)
#X_test = scaler.transform(X_test)

In [48]:
from sklearn.ensemble import RandomForestClassifier
rfc=RandomForestClassifier(n_estimators=10,random_state=42)
rfc.fit(X_train,y_train)
y_pred=rfc.predict(X_test)

In [49]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [50]:
score = accuracy_score(y_pred, y_test)
print("score:", score)
print(classification_report(y_pred, y_test))
print("confusion matrix:\n", confusion_matrix(y_pred, y_test))

score: 0.956140350877193
              precision    recall  f1-score   support

           B       1.00      0.93      0.97        76
           M       0.88      1.00      0.94        38

    accuracy                           0.96       114
   macro avg       0.94      0.97      0.95       114
weighted avg       0.96      0.96      0.96       114

confusion matrix:
 [[71  5]
 [ 0 38]]


In [51]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [52]:
model=RandomForestClassifier(random_state=42)

In [53]:
params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

In [54]:
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_grid = GridSearchCV(estimator=model,param_grid=params,cv=cv,scoring='accuracy',n_jobs=-1)
rf_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10, 20],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy')

In [55]:
print("En iyi parametreler:", rf_grid.best_params_)
y_pred = rf_grid.best_estimator_.predict(X_test)
print("score:", accuracy_score(y_pred, y_test))
print(classification_report(y_pred, y_test))
print("confusion matrix:\n", confusion_matrix(y_pred, y_test))

En iyi parametreler: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
score: 0.9649122807017544
              precision    recall  f1-score   support

           B       1.00      0.95      0.97        75
           M       0.91      1.00      0.95        39

    accuracy                           0.96       114
   macro avg       0.95      0.97      0.96       114
weighted avg       0.97      0.96      0.97       114

confusion matrix:
 [[71  4]
 [ 0 39]]
